# Vegetarian Cooking RAG chatbot

Created for the challenge in notebook 3_rag.ipynb

---
### 1. Installation and settings

In [1]:
import os
import sys
sys.path.append("../../src")
from utils.install import install_packages_if_missing

c:\Users\Darach\OneDrive\Documents\WBS_DataScience\DataFoundations\notebooks\8.GenerativeAI\veg_cooking\notebooks\../../src\utils\install.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
install_packages_if_missing(["jedi", 
                             "llama-index-core", 
                             "llama-index-llms-groq", 
                             "llama-index-readers-file", 
                             "llama-index-embeddings-huggingface", 
                             "llama-index-embeddings-instructor",
                             "pypdf", 
                             "gradio==6.3.0"])

jedi 0.19.2 already installed
llama-index-core 0.14.18 already installed
llama-index-llms-groq 0.5.0 already installed
llama-index-readers-file 0.6.0 already installed
llama-index-embeddings-huggingface 0.7.0 already installed
llama-index-embeddings-instructor 0.5.0 already installed
pypdf 6.9.2 already installed
gradio 6.3.0 already satisfies ==6.3.0

All packages are already installed and satisfy requirements.


---
### 2. Set up an LLM

In [3]:
from llama_index.llms.groq import Groq

# This info's at the top of each HuggingFace model page
model = "llama-3.3-70b-versatile"

llm = Groq(
    model=model,
    temperature = 0, #good to set low temperature (more deterministic, less creative) for RAG (between 0 and 0.4)
    api_key=os.environ.get(
        "GROQ_API_KEY"
    ),  # you can also enter your API key here, either hard-coded or read from another file
)

---
### 3. Retrieval Augmented Generation

**Prepare the data and embeddings**

In [4]:
from llama_index.core import SimpleDirectoryReader, StorageContext, load_index_from_storage
from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.prompts import PromptTemplate
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.chat_engine import ContextChatEngine
from llama_index.core.base.llms.types import ChatMessage, MessageRole

Load the documents 

In [5]:
# text_splitter = SentenceSplitter(chunk_size=1000, chunk_overlap=400)

# # load in the documents
# documents = SimpleDirectoryReader("../data", required_exts=[".pdf"]).load_data(
#     show_progress=True
# )

Initiate embeddings model

In [6]:
# embeddings
embedding_model = "sentence-transformers/all-MiniLM-L6-v2" #0.011B active parameters with a score (Mean (Task)) of 37.00
# embedding_model = "Octen/Octen-Embedding-0.6B" #0.44B active parameters with a score (Mean (Task)) of 72.41
embeddings_folder = "../storage/embedding_model/"

embeddings = HuggingFaceEmbedding(
    model_name=embedding_model, cache_folder=embeddings_folder
)

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 8bb87dab-9856-48a9-96a9-a16ccc786295)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/README.md
Retrying in 1s [Retry 1/5].


Create embedding vectors and retriever

In [7]:
# # vector database
# vector_index = VectorStoreIndex.from_documents(
#     documents, transformations=[text_splitter], embed_model=embeddings
# )

# retriever = vector_index.as_retriever(similarity_top_k=2) 

# vector_index.storage_context.persist(persist_dir="../storage/vector_index")

Or read vectors from file

In [8]:
#Load pre-saved embeddings

# ✪ different chunk sizes, overlaps, topk
storage_context = StorageContext.from_defaults(persist_dir="../storage/vector_index")
vector_index = load_index_from_storage(storage_context, embed_model=embeddings)

retriever = vector_index.as_retriever(similarity_top_k=2) 

Add system prompts

In [9]:
prefix_messages = [
    ChatMessage(
        role=MessageRole.SYSTEM,
        content="You are a nice, helpful chatbot helping a human to make vegetarian food.",
    ),
    ChatMessage(
        role=MessageRole.SYSTEM,
        content="""Answer the question based only on the following context and previous conversation. 
        Only provide a recipe if specifically asked for one. 
        Provide citations for recipes (both source and page number).""",
    ),
    ChatMessage(
        role=MessageRole.SYSTEM, content="""If the user asks for a recipe, answer in this exact format:
        <brief 2–3 sentence intro>

        Ingredients:
        - item 1
        - item 2
        - item 3

        Instructions:
        1. Step one
        2. Step two
        3. Step three
        """
    ),
]

Add memory

In [10]:
memory = ChatMemoryBuffer.from_defaults()

Initiate RAG chatbot

In [11]:
rag_bot = ContextChatEngine(
    llm=llm,
    retriever=retriever,
    memory=memory,
    prefix_messages=prefix_messages,
)

---
### 4. Create Gradio app

In [12]:
#!pip install -Uqqq gradio==6.3.0
import gradio as gr

4.1. The Custom Callback Function

The function needs to accept inputs from interactive components: the `message` and the `top_k_value` from the slider.

Because we are now building the UI manually with `gr.Blocks`, we also have to manually manage the chat history. The function will:

1.  Dynamically update the RAG retriever with the `top_k_value`.
2.  Call the `rag_bot` to retrieve the response.
3.  Return the *entire* updated history to `gr.Chatbot`.
4.  Reset the memory of the rag_bot and fill it only with the chat-history of the current user

In [13]:
def custom_rag_bot_callback(message, history, top_k_value, temperature):
    rag_bot._retriever._similarity_top_k = int(top_k_value)
    rag_bot._llm.temperature = int(temperature)

    # reset the memory and hand it only the user's own history when asking the question
    rag_bot.reset()
    print(history)
    bot_history = [
        ChatMessage(role=m["role"], content=m["content"][0]["text"])
        for m in history
    ]
    response = rag_bot.chat(message, chat_history=bot_history)

    # We append both the user and the bot message to the user's history
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response.response})

    # Return the updated list to the chatbot component
    return history

Now we can adapt the app definition accordingly:

In [14]:
with gr.Blocks(title="VeggieCook") as veggie_cook:
    gr.Markdown("<h1>Vegetarian cooking assistant 🌿</h1>")

    with gr.Row():
        with gr.Column(scale=1, min_width=250):
            gr.Markdown("## RAG Settings")
            top_k_slider = gr.Slider(minimum=1, maximum=10, step=1, value=5, 
                                     label="Number of documents (Top K)", 
                                     info = "Controls how many relevant documents are retrieved")
            temp_slider = gr.Slider(minimum=0, maximum=2, step=0.1, value=0, 
                                     label="Creativity (Temperature)", 
                                     info = "Higher values for more creative answers")

        with gr.Column(scale=4):
            chatbot = gr.Chatbot(
                label="Your veggie sous chef",
                height=500,
                value=[{"role": "assistant", "content": "Hello! 🌱 Tell me what ingredients you have and I'll suggest something delicious."}]
            )

            msg_input = gr.Textbox(label="Your Question", placeholder="Ask me about a recipe, ingredient, or technique...")

            # We pass 'chatbot' as an INPUT and an OUTPUT.
            msg_input.submit(
                fn=custom_rag_bot_callback,
                inputs=[msg_input, chatbot, top_k_slider, temp_slider],
                outputs=[chatbot]
            ).then(
                fn=lambda: "",
                outputs=[msg_input]
            )

In [15]:
veggie_theme = gr.themes.Base(
    primary_hue=gr.themes.Color(
        c50="#f2f9ee", c100="#d6eec8", c200="#aad88a", c300="#7cc55a",
        c400="#5aab2a", c500="#449020", c600="#357a18", c700="#296010",
        c800="#1e4d0c", c900="#142f07", c950="#0b1f04",
    ),
    neutral_hue=gr.themes.Color(
        c50="#f8fdf5", c100="#edf5e6", c200="#d6eac4", c300="#b8d99a",
        c400="#94c26a", c500="#72a84a", c600="#578a36", c700="#416b26",
        c800="#2e4f1a", c900="#1d3310", c950="#0e1f07",
    ),
    font=[gr.themes.GoogleFont("DM Sans"), "ui-sans-serif", "sans-serif"],
).set(
    body_background_fill="#1e4d0c",      # dark leafy green
    block_background_fill="#2e4f1a",     # slightly lighter for boxes
    input_background_fill="#243d14",
    body_text_color="#ffffff",
    block_label_text_color="#aad88a",
    input_border_color="rgba(255,255,255,0.15)",
    input_border_color_focus="#5aab2a",
    button_primary_background_fill="#5aab2a",
    button_primary_background_fill_hover="#449020",
    button_primary_text_color="#ffffff",
    block_shadow="0 8px 24px rgba(0,0,0,0.8)",
    block_radius="16px",
)

veggie_cook.launch(theme=veggie_theme, share=True, show_error=True) #https://www.gradio.app/guides/theming-guide

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [16]:
veggie_cook.close()

Closing server running on port: 7860
